In [ ]:
# Notebook 6/9 - Raw Database Construction with CERMINE (Test + Control)
#
# Purpose:
# - Extract structured sections from PDF documents (retracted/test and matched control articles)
#   using CERMINE (JATS XML output), and build/refresh a raw case-level database for downstream
#   preprocessing and analysis.
#
# What this notebook does:
# 1) Mounts Google Drive and installs dependencies required for CERMINE execution and parsing.
# 2) Uses CERMINE extraction to produce JATS XML for each PDF (test + control).
# 3) Parses JATS XML into canonical sections used throughout the dissertation pipeline.
# 4) Writes/updates a raw database CSV in a resume-safe manner (append/overwrite behaviour as implemented).
# 5) Produces basic audits and exports remaining hard failures for optional manual inspection.
#
# Inputs (Google Drive):
# - Retracted PDFs folder (test):   oa_pdfs_all/<code>.pdf
# - Control PDFs folder (control):  control_oa_pdfs_all/<code>.pdf
# - Raw database file path as configured below
#
# Outputs (Google Drive):
# - Raw database CSV containing extracted sections for test + control cases
# - Optional CSV of remaining hard failures (missing all sections)
#
# Execution:
# - Run cells top-to-bottom.
# - If interrupted, rerun; resume behaviour is controlled by the notebook’s log/database logic.
#
# Notes: This notebook is part of a larger, multi-stage data acquisition and
# analysis pipeline and is designed to be fully reproducible.

In [ ]:
# 1) Mount Google Drive + Install dependencies

from google.colab import drive
drive.mount("/content/drive")

!apt-get -qq update
# Install JDK (includes `jar` tool) + JRE
!apt-get -qq install -y default-jdk default-jre

!pip -q install pandas==2.2.2 beautifulsoup4==4.12.3 lxml==5.3.0 tqdm==4.66.5 rapidfuzz==3.9.6

In [ ]:
# 2) Imports

import os, time, random, re, requests, threading
import pandas as pd
from tqdm import tqdm
from collections import Counter
from bs4 import BeautifulSoup

In [ ]:
# 3) Paths + Download CERMINE jar

# Thread-local session so each worker keeps its own connection pool
_thread_local = threading.local()

def _get_cermine_session() -> requests.Session:
    s = getattr(_thread_local, "session", None)
    if s is None:
        s = requests.Session()
        s.headers.update({"Content-Type": "application/binary"})
        _thread_local.session = s
    return s

def cermine_extract_jats(pdf_path: str, out_dir: str, timeout_s: int = 180):
    """
    Uses CERMINE public REST service.
    Returns: (xml_path, run_note, stderr_tail)
    """
    os.makedirs(out_dir, exist_ok=True)

    if not pdf_path or not os.path.exists(pdf_path):
        return "", "cermine:missing_pdf", ""

    xml_path = os.path.join(out_dir, "cermine_jats.xml")

    # polite + resilient retries
    max_tries = 4
    base_sleep = 3.0

    url = "http://cermine.ceon.pl/extract.do"
    session = _get_cermine_session()

    try:
        with open(pdf_path, "rb") as f:
            pdf_bytes = f.read()
    except Exception as e:
        return "", f"cermine:pdf_read_error:{type(e).__name__}", str(e)[:500]

    for attempt in range(1, max_tries + 1):
        try:
            resp = session.post(url, data=pdf_bytes, timeout=timeout_s)

            if resp.status_code == 200 and resp.text and "<" in resp.text:
                with open(xml_path, "w", encoding="utf-8", errors="ignore") as out:
                    out.write(resp.text)
                return xml_path, f"cermine:rest_ok:try={attempt}", ""

            # Common transient errors: 429/ 503/ 502
            if resp.status_code in (429, 500, 502, 503, 504):
                sleep_s = base_sleep * (2 ** (attempt - 1)) + random.uniform(0, 1.5)
                time.sleep(sleep_s)
                continue

            # Hard failure
            tail = (resp.text or "")[:500]
            return "", f"cermine:rest_http_{resp.status_code}", tail

        except requests.exceptions.Timeout:
            sleep_s = base_sleep * (2 ** (attempt - 1)) + random.uniform(0, 1.5)
            time.sleep(sleep_s)
            if attempt == max_tries:
                return "", "cermine:rest_timeout", "timeout"
        except Exception as e:
            # Any other error: don't loop forever
            return "", f"cermine:rest_error:{type(e).__name__}", str(e)[:500]

    return "", "cermine:rest_failed", ""

In [ ]:
# 4) Helper functions (CERMINE REST + JATS parsing + section mapping)

# Required sections in DB
REQ_SECS = ["section_abstract","section_introduction","section_methods","section_results","section_discussion"]

# Heuristic bucket mapping for JATS <sec><title>
BUCKETS = {
    "introduction": ["introduction", "background"],
    "methods": [
        "methods", "materials and methods", "methodology", "experimental",
        "study design", "patients and methods", "statistical analysis", "data and methods"
    ],
    "results": ["results", "findings", "analysis", "outcomes"],
    "discussion": [
        "discussion", "conclusion", "conclusions", "discussion and conclusions",
        "interpretation", "limitations", "strengths and limitations"
    ],
}

EMPTY_TOKENS = {"", "nan", "none", "null", "n/a", "na", "[]", "{}", "0"}
def is_empty(x) -> bool:
    s = str(x).strip().lower()
    s = " ".join(s.split())
    return (s in EMPTY_TOKENS)

def norm_text(x: str) -> str:
    x = str(x or "").strip().lower()
    x = re.sub(r"\s+", " ", x)
    return x

# CERMINE REST endpoint must exist; if not, set a safe default
if "CERMINE_REST_URL" not in globals() or not str(CERMINE_REST_URL).strip():
    CERMINE_REST_URL = "http://cermine.ceon.pl/extract.do"

def cermine_extract_jats(pdf_path: str, out_dir: str, timeout_s: int = 180):
    """
    Uses CERMINE public REST service.
    Returns: (xml_path, run_note, stderr_tail)
    """
    os.makedirs(out_dir, exist_ok=True)

    if not pdf_path or not os.path.exists(pdf_path):
        return "", "cermine:missing_pdf", ""

    xml_path = os.path.join(out_dir, "cermine_jats.xml")

    max_tries = 4
    base_sleep = 3.0
    headers = {"Content-Type": "application/binary"}

    for attempt in range(1, max_tries + 1):
        try:
            with open(pdf_path, "rb") as f:
                resp = requests.post(
                    CERMINE_REST_URL,
                    data=f,
                    headers=headers,
                    timeout=timeout_s
                )

            if resp.status_code == 200 and resp.text and "<" in resp.text:
                with open(xml_path, "w", encoding="utf-8", errors="ignore") as out:
                    out.write(resp.text)
                return xml_path, f"cermine:rest_ok:try={attempt}", ""

            # transient errors
            if resp.status_code in (429, 500, 502, 503, 504):
                sleep_s = base_sleep * (2 ** (attempt - 1)) + random.uniform(0, 1.5)
                time.sleep(sleep_s)
                continue

            tail = (resp.text or "")[:500]
            return "", f"cermine:rest_http_{resp.status_code}", tail

        except requests.exceptions.Timeout:
            sleep_s = base_sleep * (2 ** (attempt - 1)) + random.uniform(0, 1.5)
            time.sleep(sleep_s)
            if attempt == max_tries:
                return "", "cermine:rest_timeout", "timeout"
        except Exception as e:
            return "", f"cermine:rest_error:{type(e).__name__}", str(e)[:500]

    return "", "cermine:rest_failed", ""

def parse_jats_sections(xml_path: str):
    """
    Parse JATS XML from CERMINE and extract:
    abstract/introduction/methods/results/discussion.

    Returns: (sections_dict, parse_note)
    """
    out = {k: "" for k in ["abstract", "introduction", "methods", "results", "discussion"]}

    if not xml_path or not os.path.exists(xml_path):
        return out, "cermine:no_xml"

    try:
        with open(xml_path, "r", encoding="utf-8", errors="ignore") as f:
            xml = f.read()

        soup = BeautifulSoup(xml, "xml")

        # Abstract
        abs_node = soup.find("abstract")
        if abs_node:
            out["abstract"] = abs_node.get_text("\n", strip=True)

        # Sections (prefer <body>, fallback to all <sec>)
        body = soup.find("body")
        sec_nodes = body.find_all("sec") if body else soup.find_all("sec")
        if not sec_nodes:
            return out, "cermine:no_secs"

        for sec in sec_nodes:
            title_node = sec.find("title")
            title = norm_text(title_node.get_text(" ", strip=True) if title_node else "")
            if not title:
                continue

            chosen = None
            for canon, keys in BUCKETS.items():
                if any(k in title for k in keys):
                    chosen = canon
                    break
            if not chosen:
                continue

            txt = sec.get_text("\n", strip=True)
            if txt:
                out[chosen] = (out[chosen] + "\n" + txt).strip()

        return out, "cermine:ok"

    except Exception as e:
        return out, f"cermine:error:{type(e).__name__}"

In [ ]:
# 5) Load DB + build retry list + DEFINE ALL PATHS + CHECKPOINT HELPERS (order-safe)

# 5.1) BASE PATHS
BASE_DIR = "/content/drive/MyDrive/Dissertation"
RAW_DB_CSV = os.path.join(BASE_DIR, "raw_fulltext_db.csv")

# Define CERMINE output folder
CERMINE_OUT_DIR = os.path.join(BASE_DIR, "cermine_outputs")
os.makedirs(CERMINE_OUT_DIR, exist_ok=True)

# Define audit CSV (append-only)
CERMINE_AUDIT_CSV = os.path.join(BASE_DIR, "cermine_audit_log.csv")
os.makedirs(os.path.dirname(CERMINE_AUDIT_CSV), exist_ok=True)

# 5.2) CHECKPOINT SETTINGS (used by run_subset)
AUDIT_FLUSH_EVERY = 25
DB_FLUSH_EVERY    = 25

# 5.3) REQUIRED SECTIONS
REQ_SECS = ["section_abstract","section_introduction","section_methods","section_results","section_discussion"]

if not os.path.exists(RAW_DB_CSV):
    raise FileNotFoundError(f"RAW DB not found: {RAW_DB_CSV}")

db = pd.read_csv(RAW_DB_CSV, dtype=str).fillna("")

# Ensure required columns exist
for c in REQ_SECS + ["paper_id", "extraction_source", "extraction_notes", "cermine_attempted"]:
    if c not in db.columns:
        db[c] = ""

# Normalize attempted flag
db["cermine_attempted"] = (
    db["cermine_attempted"]
    .astype(str)
    .replace({"": "0", "nan": "0", "None": "0"})
    .fillna("0")
)

# Index for fast updates (used by run_subset)
db_idx = db.set_index("paper_id", drop=False)

# 5.4) PDF path column handling
PDF_PATH_CANDIDATES = ["source_pdf_path", "pdf_path", "pdf_file", "local_pdf_path", "pdf_local_path"]
def get_pdf_path(row):
    for col in PDF_PATH_CANDIDATES:
        if col in row and str(row[col]).strip():
            return str(row[col]).strip()
    return ""

# Robust emptiness check
EMPTY_TOKENS = {"", "nan", "none", "null", "n/a", "na", "[]", "{}", "0"}
def is_empty(x) -> bool:
    s = str(x).strip().lower()
    s = " ".join(s.split())
    return (s in EMPTY_TOKENS)

# 5.5) Crash-safe audit flushing (append mode)
def flush_audit_rows(audit_buffer):
    if not audit_buffer:
        return
    df = pd.DataFrame(audit_buffer)
    write_header = (not os.path.exists(CERMINE_AUDIT_CSV)) or (os.path.getsize(CERMINE_AUDIT_CSV) == 0)
    df.to_csv(CERMINE_AUDIT_CSV, mode="a", header=write_header, index=False)
    audit_buffer.clear()

# 5.6) Crash-safe DB checkpoint (overwrite RAW_DB_CSV)
def checkpoint_db():
    db_idx.reset_index(drop=True).to_csv(RAW_DB_CSV, index=False)

# 5.7) Compute missing flags
missing_any_required = db[REQ_SECS].apply(lambda r: any(is_empty(r[c]) for c in REQ_SECS), axis=1)
missing_all_required = db[REQ_SECS].apply(lambda r: all(is_empty(r[c]) for c in REQ_SECS), axis=1)

db["needs_cermine_retry"] = (missing_any_required | missing_all_required).astype(int)
db["missing_all_sections"] = missing_all_required.astype(int)

def missing_list(row):
    miss = [c.replace("section_","") for c in REQ_SECS if is_empty(row[c])]
    return ",".join(miss)

db["missing_sections_list"] = db.apply(missing_list, axis=1)

retry_df = db[(missing_any_required | missing_all_required)].copy()

print("BASE_DIR:", BASE_DIR)
print("RAW_DB_CSV:", RAW_DB_CSV)
print("CERMINE_OUT_DIR:", CERMINE_OUT_DIR)
print("CERMINE_AUDIT_CSV:", CERMINE_AUDIT_CSV)
print("RAW DB rows:", len(db))
print("Retry rows (missing ANY required section):", int(missing_any_required.sum()))
print("Retry rows (missing ALL required sections):", int(missing_all_required.sum()))
print("Total retry rows (union):", len(retry_df))

# Persist flags back (optional but helpful)
checkpoint_db()
print("RAW DB checkpointed (with flags):", RAW_DB_CSV)

In [ ]:
# 6) Run BOTH passes + diagnostics + UPDATE QUALITY + RESUME + CHECKPOINT LOGS

# 6.1) CHECKPOINT SETTINGS
AUDIT_FLUSH_EVERY = 25
DB_FLUSH_EVERY    = 25

# 6.2) Ensure required PATH globals exist
BASE_DIR = os.path.dirname(RAW_DB_CSV)

# Define CERMINE_OUT_DIR if earlier cells didn't (prevents NameError)
if "CERMINE_OUT_DIR" not in globals() or not str(CERMINE_OUT_DIR).strip():
    CERMINE_OUT_DIR = os.path.join(BASE_DIR, "cermine_outputs")
os.makedirs(CERMINE_OUT_DIR, exist_ok=True)

# Define audit CSV if earlier cells didn't
if "CERMINE_AUDIT_CSV" not in globals() or not str(CERMINE_AUDIT_CSV).strip():
    CERMINE_AUDIT_CSV = os.path.join(BASE_DIR, "cermine_audit_log.csv")
os.makedirs(os.path.dirname(CERMINE_AUDIT_CSV), exist_ok=True)

print("CERMINE_OUT_DIR:", CERMINE_OUT_DIR)
print("CERMINE_AUDIT_CSV:", CERMINE_AUDIT_CSV)

# 6.3) Ensure columns exist (including resume flag)
NEEDED_COLS = REQ_SECS + [
    "paper_id",
    "extraction_source",
    "extraction_notes",
    "quality_percent",
    "confidence_percent",
    "quality_rationale",
    "cermine_attempted",
]
for c in NEEDED_COLS:
    if c not in db.columns:
        db[c] = ""

db["cermine_attempted"] = (
    db["cermine_attempted"]
    .astype(str)
    .replace({"": "0", "nan": "0", "None": "0"})
    .fillna("0")
)

db_idx = db.set_index("paper_id", drop=False)

# 6.4) Quality scoring
def quality_score(sections, source_note=""):
    lens = {k: len((v or "").strip()) for k, v in sections.items()}
    non_empty = sum(
        1 for k in ["abstract", "introduction", "methods", "results", "discussion"]
        if lens.get(k, 0) >= 200
    )

    score = 10 + non_empty * 18
    used_web = isinstance(source_note, str) and (
        source_note.startswith("pmc:") or source_note.startswith("unpaywall:") or source_note.startswith("openalex:")
    )
    used_grobid = isinstance(source_note, str) and source_note.startswith("grobid:")

    if used_web:
        score += 10
    if used_grobid:
        score += 15

    for k in ["introduction", "methods", "results", "discussion"]:
        if lens.get(k, 0) < 500:
            score -= 10

    score = max(0, min(100, score))

    conf = 55
    heading_detected = any(lens.get(k, 0) > 0 for k in ["abstract", "methods", "results", "discussion"])
    if heading_detected:
        conf += 20
    if used_web:
        conf += 15
    if used_grobid:
        conf += 20
    if non_empty >= 4:
        conf += 10

    conf = max(0, min(100, conf))

    rationale = f"non_empty_sections={non_empty}/5; heading_detected={heading_detected}; source={source_note}; lens={lens}"
    return str(score), str(conf), rationale

def recompute_quality_for_row(row):
    secs = {
        "abstract": row.get("section_abstract", ""),
        "introduction": row.get("section_introduction", ""),
        "methods": row.get("section_methods", ""),
        "results": row.get("section_results", ""),
        "discussion": row.get("section_discussion", ""),
    }
    src = str(row.get("extraction_source", "") or "")
    q, conf, why = quality_score(secs, source_note=src)
    return q, conf, why

# 6.5) Crash-safe audit flushing (append mode)
def flush_audit_rows(audit_buffer):
    """Append audit rows to disk frequently to survive crashes."""
    if not audit_buffer:
        return
    os.makedirs(os.path.dirname(CERMINE_AUDIT_CSV), exist_ok=True)
    df = pd.DataFrame(audit_buffer)
    write_header = not os.path.exists(CERMINE_AUDIT_CSV) or os.path.getsize(CERMINE_AUDIT_CSV) == 0
    df.to_csv(CERMINE_AUDIT_CSV, mode="a", header=write_header, index=False)
    audit_buffer.clear()

# 6.6) Crash-safe DB checkpoint (overwrite RAW_DB_CSV)
def checkpoint_db():
    """
    Save current db_idx state to RAW_DB_CSV so cermine_attempted + filled sections
    survive crashes. (Flags/quality are finalized at the end.)
    """
    tmp = db_idx.reset_index(drop=True)
    tmp.to_csv(RAW_DB_CSV, index=False)

# 6.7) Runner for a subset
def run_subset(sub_df, pass_name, diag_limit=8):
    updated_count = 0
    no_improve_count = 0
    fail_count = 0
    missing_pdf_count = 0
    skipped_count = 0

    fail_notes = Counter()
    sample_failures = []
    audit_buffer = []

    processed_in_pass = 0

    for _, r in tqdm(sub_df.iterrows(), total=len(sub_df), desc=f"CERMINE {pass_name}", unit="paper"):
        paper_id = str(r.get("paper_id", "")).strip()
        if not paper_id or paper_id not in db_idx.index:
            skipped_count += 1
            continue

        # RESUME GUARD
        if str(db_idx.at[paper_id, "cermine_attempted"]).strip() == "1":
            skipped_count += 1
            continue

        # Mark attempted immediately so restarts won't repeat this paper
        db_idx.at[paper_id, "cermine_attempted"] = "1"

        pdf_path = get_pdf_path(r)
        if not pdf_path or not os.path.exists(pdf_path):
            missing_pdf_count += 1
            db_idx.at[paper_id, "extraction_notes"] = "cermine:missing_pdf"

            audit_buffer.append({
                "paper_id": paper_id,
                "status": "missing_pdf",
                "pass": pass_name,
                "pdf_path": pdf_path,
                "missing_sections_before": str(r.get("missing_sections_list", ""))
            })
        else:
            out_dir = os.path.join(CERMINE_OUT_DIR, paper_id.replace(":", "_"))

            # REST extractor (Cell 2 defines this signature)
            xml_path, run_note, stderr_tail = cermine_extract_jats(pdf_path, out_dir, timeout_s=180)
            secs, parse_note = parse_jats_sections(xml_path)

            if parse_note != "cermine:ok":
                fail_count += 1
                key = f"{run_note}|{parse_note}"
                fail_notes[key] += 1
                db_idx.at[paper_id, "extraction_notes"] = key

                if len(sample_failures) < diag_limit:
                    sample_failures.append({
                        "paper_id": paper_id,
                        "run_note": run_note,
                        "parse_note": parse_note,
                        "pdf_path": pdf_path,
                        "xml_path": xml_path,
                        "stderr_tail": stderr_tail
                    })

                audit_buffer.append({
                    "paper_id": paper_id,
                    "status": "fail",
                    "pass": pass_name,
                    "run_note": run_note,
                    "parse_note": parse_note,
                    "pdf_path": pdf_path,
                    "xml_path": xml_path,
                    "stderr_tail": stderr_tail,
                    "missing_sections_before": str(r.get("missing_sections_list", ""))
                })
            else:
                fill_map = {
                    "section_abstract": secs.get("abstract", ""),
                    "section_introduction": secs.get("introduction", ""),
                    "section_methods": secs.get("methods", ""),
                    "section_results": secs.get("results", ""),
                    "section_discussion": secs.get("discussion", ""),
                }

                filled_cols = []
                for col, val in fill_map.items():
                    if is_empty(db_idx.at[paper_id, col]) and not is_empty(val):
                        db_idx.at[paper_id, col] = val
                        filled_cols.append(col)

                if filled_cols:
                    updated_count += 1
                    prev_src = str(db_idx.at[paper_id, "extraction_source"]).strip()
                    db_idx.at[paper_id, "extraction_source"] = (prev_src + "|cermine").strip("|")
                    db_idx.at[paper_id, "extraction_notes"] = "cermine:filled_missing"

                    audit_buffer.append({
                        "paper_id": paper_id,
                        "status": "updated",
                        "pass": pass_name,
                        "filled_cols": ",".join(filled_cols),
                        "pdf_path": pdf_path,
                        "xml_path": xml_path,
                        "missing_sections_before": str(r.get("missing_sections_list", ""))
                    })
                else:
                    no_improve_count += 1
                    db_idx.at[paper_id, "extraction_notes"] = "cermine:ok_but_no_improvement"

                    audit_buffer.append({
                        "paper_id": paper_id,
                        "status": "no_improvement",
                        "pass": pass_name,
                        "pdf_path": pdf_path,
                        "xml_path": xml_path,
                        "missing_sections_before": str(r.get("missing_sections_list", ""))
                    })

        processed_in_pass += 1

        # Flush audit frequently
        if processed_in_pass % AUDIT_FLUSH_EVERY == 0:
            flush_audit_rows(audit_buffer)

        # Checkpoint DB frequently (so resume survives crashes)
        if processed_in_pass % DB_FLUSH_EVERY == 0:
            checkpoint_db()

    # Final flush for this pass
    flush_audit_rows(audit_buffer)
    checkpoint_db()

    print(f"\n=== SUMMARY: {pass_name} ===")
    print("Updated (filled >=1 section):", updated_count)
    print("OK but no improvement:", no_improve_count)
    print("Failures:", fail_count)
    print("Missing PDFs:", missing_pdf_count)
    print("ℹSkipped (already attempted / invalid):", skipped_count)
    print("Audit checkpointed to:", CERMINE_AUDIT_CSV)
    print("DB checkpointed to:", RAW_DB_CSV)

    if fail_count > 0:
        print("\nTop failure signatures (run_note|parse_note):")
        for k, v in fail_notes.most_common(5):
            print(f"  {v}  ->  {k}")

        print("\nSample failure details:")
        for s in sample_failures:
            print("----")
            print("paper_id:", s["paper_id"])
            print("run_note:", s["run_note"])
            print("parse_note:", s["parse_note"])
            print("xml_path:", s["xml_path"])
            print("stderr_tail:", (s["stderr_tail"] or "")[:500])

# 6.8) Build subsets — RESUME-AWARE (and compatible with older retry_df)
if "cermine_attempted" not in retry_df.columns:
    retry_df = retry_df.copy()
    retry_df["cermine_attempted"] = "0"

# Order: PARTIAL first, then ALL
partial_missing_df = retry_df[
    (retry_df["missing_all_sections"].astype(str).ne("1")) &
    (retry_df["cermine_attempted"].astype(str).ne("1"))
].copy()

all_missing_df = db[
    (db["missing_all_sections"].astype(str).eq("1")) &
    (db["cermine_attempted"].astype(str).ne("1"))
].copy()

print("PASS B (missing SOME required, not attempted):", len(partial_missing_df))
print("PASS A (missing ALL required, not attempted):", len(all_missing_df))
print("Total to process (resume-aware):", len(partial_missing_df) + len(all_missing_df))
print("Audit CSV (will append):", CERMINE_AUDIT_CSV)

# 6.9) Run passes
run_subset(partial_missing_df, "PASS_B_PARTIAL_MISSING", diag_limit=8)
run_subset(all_missing_df, "PASS_A_ALL_MISSING", diag_limit=8)

# 6.10) Finalize: recompute flags + quality/confidence (expensive, do once at end)
db_out = db_idx.reset_index(drop=True)

missing_any_after = db_out[REQ_SECS].apply(lambda rr: any(is_empty(rr[c]) for c in REQ_SECS), axis=1)
missing_all_after = db_out[REQ_SECS].apply(lambda rr: all(is_empty(rr[c]) for c in REQ_SECS), axis=1)

db_out["needs_cermine_retry"] = (missing_any_after | missing_all_after).astype(int)
db_out["missing_all_sections"] = missing_all_after.astype(int)
db_out["missing_sections_list"] = db_out.apply(
    lambda row: ",".join([c.replace("section_", "") for c in REQ_SECS if is_empty(row[c])]),
    axis=1
)

qc = db_out.apply(lambda r: recompute_quality_for_row(r), axis=1)
db_out["quality_percent"] = [x[0] for x in qc]
db_out["confidence_percent"] = [x[1] for x in qc]
db_out["quality_rationale"] = [x[2] for x in qc]

db_out.to_csv(RAW_DB_CSV, index=False)
print("\nFINAL save RAW DB (flags + quality/conf refreshed):", RAW_DB_CSV)
print("FINAL audit CSV:", CERMINE_AUDIT_CSV)

In [ ]:
# 7) Post-run audit summary

db2 = pd.read_csv(RAW_DB_CSV, dtype=str).fillna("")

missing_any = db2[REQ_SECS].apply(lambda r: any(str(r[c]).strip()=="" for c in REQ_SECS), axis=1)
missing_all = db2[REQ_SECS].apply(lambda r: all(str(r[c]).strip()=="" for c in REQ_SECS), axis=1)

print("After CERMINE:")
print("Total rows:", len(db2))
print("Missing ANY required section:", int(missing_any.sum()))
print("Missing ALL required sections:", int(missing_all.sum()))

# Optional: show top examples still missing all
still_all = db2[missing_all].copy()
print("\nExamples still missing ALL sections (first 10 paper_id):")
print(still_all["paper_id"].head(10).tolist())